# 🤖 Agentic AI System
This notebook sets up an agentic AI that can **search the web** or **write & run Python code** based on your query.

**Steps:** Install → Write agent.py → Write app.py → Launch Streamlit via ngrok

In [ ]:
!pip install tavily-python groq streamlit python-dotenv pyngrok

In [ ]:
%%writefile agent.py
import os
from tavily import TavilyClient
from groq import Groq
import subprocess
import tempfile
import re

# ✅ Set your API keys here
os.environ["TAVILY_API_KEY"] = "Add Your Key fron tavily"
os.environ["GROQ_API_KEY"] = "Add Your key from groq"

# Initialize APIs
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])


# ---------------------------
# 1. PLANNER
# ---------------------------
def planner(query):
    keywords = ["calculate", "sum", "multiply", "interest", "python", "code", "compute", "solve"]
    for word in keywords:
        if word in query.lower():
            return "code"
    return "search"


# ---------------------------
# 2. SEARCH TOOL (Tavily)
# ---------------------------
def search_tool(query):
    try:
        result = tavily.search(query=query, max_results=3)
        content = ""
        sources = []
        for r in result["results"]:
            content += r["content"] + "\n"
            sources.append(r["url"])
        return content, sources
    except Exception as e:
        return f"Search failed: {str(e)}", []


# ---------------------------
# 3. CODE WRITER (Groq)
# ---------------------------
def extract_code(text):
    """Extract pure Python code from LLM response (removes ```python ... ``` fences)"""
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()


def code_writer(query):
    prompt = (
        f"Write ONLY executable Python code for this problem (no explanation, no markdown fences):\n{query}\n"
        "Output just the raw Python code."
    )
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    return extract_code(raw)  # FIX: clean the code before returning


# ---------------------------
# 4. CODE EXECUTOR
# ---------------------------
def execute_code(code):
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".py", mode="w") as f:
            f.write(code)
            file_path = f.name

        result = subprocess.run(
            ["python3", file_path],
            capture_output=True,
            text=True,
            timeout=30  # FIX: added timeout to prevent hanging
        )
        os.unlink(file_path)  # FIX: clean up temp file after execution
        return result.stdout, result.stderr

    except subprocess.TimeoutExpired:
        return "", "Error: Code execution timed out after 30 seconds."
    except Exception as e:
        return "", str(e)


# ---------------------------
# 5. RESPONSE GENERATOR
# ---------------------------
def generate_response(query):
    decision = planner(query)

    if decision == "search":
        content, sources = search_tool(query)
        return {
            "answer": content,
            "code": "",
            "execution_output": "",
            "sources": sources
        }
    else:
        code = code_writer(query)
        output, error = execute_code(code)
        return {
            "answer": output if output else error,
            "code": code,
            "execution_output": output if output else error,
            "sources": []
        }


In [ ]:
%%writefile app.py
import streamlit as st
import importlib
import agent

# 🔥 Force reload on every run (important in notebooks)
importlib.reload(agent)

st.title("🤖 Agentic AI System")
st.markdown("Ask a question — the agent will **search the web** or **write & run code** based on your query.")

query = st.text_input("Ask your question:")

if st.button("Run Agent"):
    if query.strip():  # FIX: check for non-empty query
        with st.spinner("Thinking..."):  # FIX: added spinner for UX
            result = agent.generate_response(query)

        st.subheader("✅ Answer")
        st.write(result["answer"])

        if result["code"]:
            st.subheader("💻 Generated Code")
            st.code(result["code"], language="python")

        if result["execution_output"]:
            st.subheader("📊 Execution Output")
            st.text(result["execution_output"])

        if result["sources"]:
            st.subheader("🔗 Sources")
            for s in result["sources"]:
                st.write(s)
    else:
        st.warning("Please enter a question first.")  # FIX: handle empty input


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3CylURT31BXcKplX1xmKfSqvV4E_2fR9xgkmubxvVFQnUj2MX")

In [ ]:
import subprocess, time

# Kill any existing streamlit process first
subprocess.run(["pkill", "-9", "-f", "streamlit"], capture_output=True)
time.sleep(2)  # Wait for port to free up

# Start Streamlit in background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(3)  # Wait for Streamlit to start

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print("✅ App is live at:", public_url)


## ✅ Done!
Click the URL above to open your Agentic AI app in the browser.